# Complete Workflow: Library Access Equity Study

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/06-complete-workflow.ipynb)

This notebook demonstrates a complete SocialMapper analysis:

**Research Question:** How equitable is library access in a city? Who has walkable access to public libraries?

**Workflow:**
1. Create walking isochrone from downtown
2. Find all libraries in the area
3. Get census blocks in the reachable zone
4. Retrieve demographic data
5. Create visualization
6. Generate summary statistics

## Setup

In [ ]:
!pip install -q socialmapper[routing]

In [ ]:
import os
import json
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map
)
from IPython.display import Image, display

print("Ready for analysis!")

## Step 1: Define the Study Area

In [ ]:
# Study location
location = "Portland, OR"

# Create a 20-minute walking isochrone
walk_isochrone = create_isochrone(
    location=location,
    travel_time=20,
    travel_mode="walk"
)

print(f"Study Area: {location}")
print(f"Travel Mode: Walking")
print(f"Travel Time: 20 minutes")
print(f"Area Coverage: {walk_isochrone['properties']['area_sq_km']:.2f} km²")

## Step 2: Find Libraries

In [ ]:
# Query for public libraries
libraries = get_poi(
    location=location,
    categories=["library"],
    travel_time=20,
    limit=50
)

print(f"Libraries found: {len(libraries)}")
print("\nLibrary Details:")
for lib in libraries:
    print(f"  - {lib['name']}: {lib['distance_km']:.2f} km from center")

## Step 3: Get Census Geography

In [ ]:
# Get census blocks within the walkable area
blocks = get_census_blocks(polygon=walk_isochrone)

print(f"Census block groups in study area: {len(blocks)}")

# Show summary
total_area = sum(b['area_sq_km'] for b in blocks)
print(f"Total census area: {total_area:.2f} km²")

# Show first few
print("\nSample block groups:")
for block in blocks[:3]:
    print(f"  GEOID: {block['geoid']}, Area: {block['area_sq_km']:.2f} km²")

## Step 4: Retrieve Demographics

In [ ]:
# Get demographic data for all blocks
geoids = [b['geoid'] for b in blocks]

census_result = get_census_data(
    location=geoids,
    variables=["population", "median_income", "median_age"]
)

print(f"Census data retrieved:")
print(f"  Year: {census_result.query_info['year']}")
print(f"  Block groups: {len(census_result.data)}")

# Combine census data with blocks
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0
    block['median_income'] = data.get('median_income', 0) or 0
    block['median_age'] = data.get('median_age', 0) or 0

## Step 5: Analyze the Data

In [ ]:
# Calculate statistics
total_population = sum(b['population'] for b in blocks)
populations = [b['population'] for b in blocks if b['population'] > 0]
incomes = [b['median_income'] for b in blocks if b['median_income'] > 0]
ages = [b['median_age'] for b in blocks if b['median_age'] > 0]

print("=" * 50)
print("STUDY RESULTS: Library Access Equity Analysis")
print("=" * 50)

print(f"\nGeographic Coverage:")
print(f"  Walkable area: {walk_isochrone['properties']['area_sq_km']:.2f} km²")
print(f"  Census block groups: {len(blocks)}")
print(f"  Libraries accessible: {len(libraries)}")

print(f"\nPopulation with Library Access:")
print(f"  Total population: {total_population:,}")
print(f"  Average per block group: {total_population // len(blocks):,}")

if incomes:
    print(f"\nEconomic Profile:")
    print(f"  Median income range: ${min(incomes):,} - ${max(incomes):,}")
    print(f"  Average median income: ${sum(incomes)//len(incomes):,}")

if ages:
    print(f"\nAge Profile:")
    print(f"  Median age range: {min(ages):.1f} - {max(ages):.1f} years")
    print(f"  Average median age: {sum(ages)/len(ages):.1f} years")

## Step 6: Create Visualizations

In [ ]:
# Filter blocks with population data
blocks_with_pop = [b for b in blocks if b['population'] > 0]

# Create population choropleth
pop_map = create_map(
    data=blocks_with_pop,
    column="population",
    title="Population with Walkable Library Access",
    save_path="library_access_population.png"
)

print(f"Population map saved: {pop_map.file_path}")
display(Image(filename="library_access_population.png"))

In [ ]:
# Create income choropleth
blocks_with_income = [b for b in blocks if b['median_income'] > 0]

income_map = create_map(
    data=blocks_with_income,
    column="median_income",
    title="Median Income in Library-Accessible Areas",
    save_path="library_access_income.png"
)

print(f"Income map saved: {income_map.file_path}")
display(Image(filename="library_access_income.png"))

## Step 7: Export Data

In [ ]:
# Export as GeoJSON for web mapping
geojson_result = create_map(
    data=blocks_with_pop,
    column="population",
    export_format="geojson"
)

with open("library_access.geojson", "w") as f:
    json.dump(geojson_result.geojson_data, f, indent=2)

print("GeoJSON exported: library_access.geojson")

In [ ]:
# Create summary report
report = {
    "study": "Library Access Equity Analysis",
    "location": location,
    "travel_mode": "walk",
    "travel_time_minutes": 20,
    "coverage": {
        "area_sq_km": walk_isochrone['properties']['area_sq_km'],
        "block_groups": len(blocks),
        "libraries": len(libraries)
    },
    "demographics": {
        "total_population": total_population,
        "avg_median_income": sum(incomes)//len(incomes) if incomes else None,
        "avg_median_age": round(sum(ages)/len(ages), 1) if ages else None
    },
    "libraries": [
        {"name": lib['name'], "distance_km": round(lib['distance_km'], 2)}
        for lib in libraries
    ]
}

with open("library_access_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Report saved: library_access_report.json")
print("\nReport contents:")
print(json.dumps(report, indent=2))

## Complete Analysis Function

Here's the full workflow as a reusable function:

In [ ]:
def analyze_library_access(location: str, travel_time: int = 20):
    """
    Analyze library accessibility for a given location.
    
    Parameters
    ----------
    location : str
        City name (e.g., "Portland, OR")
    travel_time : int
        Walking time in minutes
    
    Returns
    -------
    dict
        Analysis results
    """
    print(f"\n{'='*60}")
    print(f"Library Access Equity Analysis: {location}")
    print(f"{'='*60}")
    
    # Step 1: Create walking isochrone
    print("\n[1/6] Creating walking isochrone...")
    isochrone = create_isochrone(
        location=location,
        travel_time=travel_time,
        travel_mode="walk"
    )
    print(f"      Area: {isochrone['properties']['area_sq_km']:.2f} km²")
    
    # Step 2: Find libraries
    print("\n[2/6] Finding libraries...")
    libraries = get_poi(
        location=location,
        categories=["library"],
        travel_time=travel_time,
        limit=50
    )
    print(f"      Found: {len(libraries)} libraries")
    
    # Step 3: Get census blocks
    print("\n[3/6] Getting census blocks...")
    blocks = get_census_blocks(polygon=isochrone)
    print(f"      Block groups: {len(blocks)}")
    
    # Step 4: Get demographics
    print("\n[4/6] Retrieving demographics...")
    geoids = [b['geoid'] for b in blocks]
    census_result = get_census_data(
        location=geoids,
        variables=["population", "median_income", "median_age"]
    )
    print(f"      Data retrieved for {len(census_result.data)} blocks")
    
    # Step 5: Process data
    print("\n[5/6] Processing data...")
    for block in blocks:
        data = census_result.data.get(block['geoid'], {})
        block['population'] = data.get('population', 0) or 0
        block['median_income'] = data.get('median_income', 0) or 0
        block['median_age'] = data.get('median_age', 0) or 0
    
    total_pop = sum(b['population'] for b in blocks)
    incomes = [b['median_income'] for b in blocks if b['median_income'] > 0]
    ages = [b['median_age'] for b in blocks if b['median_age'] > 0]
    
    # Step 6: Create maps
    print("\n[6/6] Creating visualizations...")
    blocks_valid = [b for b in blocks if b['population'] > 0]
    
    filename = f"{location.replace(', ', '_').replace(' ', '_').lower()}_library_pop.png"
    pop_map = create_map(
        data=blocks_valid,
        column="population",
        title=f"Population with Library Access - {location}",
        save_path=filename
    )
    print(f"      Saved: {filename}")
    
    # Summary
    print(f"\n{'='*60}")
    print("RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Location: {location}")
    print(f"Travel time: {travel_time} minutes walking")
    print(f"Area covered: {isochrone['properties']['area_sq_km']:.2f} km²")
    print(f"Libraries: {len(libraries)}")
    print(f"Block groups: {len(blocks)}")
    print(f"Total population with access: {total_pop:,}")
    if incomes:
        print(f"Average median income: ${sum(incomes)//len(incomes):,}")
    if ages:
        print(f"Average median age: {sum(ages)/len(ages):.1f} years")
    
    return {
        "location": location,
        "travel_time": travel_time,
        "isochrone": isochrone,
        "libraries": libraries,
        "blocks": blocks,
        "total_population": total_pop,
        "avg_income": sum(incomes)//len(incomes) if incomes else None,
        "avg_age": sum(ages)/len(ages) if ages else None,
        "map_file": filename
    }

In [ ]:
# Run the complete analysis
results = analyze_library_access("Seattle, WA", travel_time=20)

# Display the map
display(Image(filename=results['map_file']))

## Comparing Multiple Cities

In [ ]:
cities = ["Portland, OR", "Seattle, WA", "San Francisco, CA"]

comparison = {}
for city in cities:
    result = analyze_library_access(city, travel_time=20)
    comparison[city] = {
        "population": result["total_population"],
        "libraries": len(result["libraries"]),
        "avg_income": result["avg_income"]
    }

# Summary comparison
print("\n" + "="*60)
print("CITY COMPARISON")
print("="*60)

for city, data in comparison.items():
    print(f"\n{city}:")
    print(f"  Population with library access: {data['population']:,}")
    print(f"  Libraries: {data['libraries']}")
    if data['avg_income']:
        print(f"  Avg income: ${data['avg_income']:,}")

## Next Steps

- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Apply these techniques to food access analysis
- Try analyzing different amenities (hospitals, schools, parks)
- Compare urban vs. suburban accessibility